In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
cat_cols = ["gender", "stress_level", "academic_work_impact"]
num_cols = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
            "work_study_hours", "sleep_hours", "notifications_per_day",
            "app_opens_per_day", "weekend_screen_time"]

In [3]:
numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median"))
])

categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [4]:
preprocessor = ColumnTransformer([
    ("num", numeric_branch, num_cols),
    ("cat", categorical_branch, cat_cols)
])

In [5]:
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import pandas as pd

df = pd.read_csv("data/train.csv")

X = df.drop(columns=["id", "addicted_label"])
y = df["addicted_label"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)

print("Fold AUCs:", np.round(scores, 5))
print(f"Mean AUC : {scores.mean():.5f}  +/- {scores.std():.5f}")

Fold AUCs: [0.91035 0.9108  0.91188 0.91265 0.91158]
Mean AUC : 0.91145  +/- 0.00081
